# Paso 1, antes de entrenar: barrido del learning rate

Busca el **LR más alto que no hace olvidar español general**, con ensayos cortos del paso 1 (mismo corpus, misma LoRA, mismo lote; `PASOS_ENSAYO` pasos con calentamiento y coseno completos).

**Por qué búsqueda binaria sobre el olvido y no sobre la pérdida.** La pérdida en función del LR tiene forma de U (con LR bajo no aprende, con LR alto diverge), así que no se puede bisecar. Lo que sí crece con el LR es el olvido. Un ensayo es **aceptable** si:
- la perplejidad del corpus de validación baja (aprende y no diverge), y
- la perplejidad de las frases **neutras** del dataset sube a lo más `UMBRAL_OLVIDO` (no olvida el registro que después tiene que producir).

Se busca en escala logarítmica entre `LR_MIN` y `LR_MAX`: primero `LR_MAX` (si ya es aceptable, no hace falta buscar), luego `ITERACIONES` bisecciones. Se reporta también cuánto baja la perplejidad de las frases **chilenas** (lo que queremos ganar).

**Límite conocido**: un ensayo corto olvida menos que el entrenamiento completo (~150 pasos), así que el LR elegido es un techo optimista. El entrenamiento completo debería vigilar las frases neutras igual.

El modelo se carga una vez; cada ensayo agrega una LoRA nueva y al final la quita sin fusionarla (`unload`). Cada ensayo se guarda en el Hub al terminar: si la sesión se corta, al volver a ejecutar no se repiten.

In [ ]:
%pip install -q -U "transformers>=5.5" "peft>=0.17" accelerate datasets huggingface_hub sacrebleu
%pip uninstall -q -y torchao   # el torchao de Colab (0.10) rompe a peft al crear el LoRA; no se usa

In [ ]:
import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
p = torch.cuda.get_device_properties(0)
VRAM_GB = p.total_memory / 1e9
print(f"GPU: {p.name} | {VRAM_GB:.0f} GB | bf16: {torch.cuda.is_bf16_supported()}")

In [ ]:
# Configuración
REPO_HF = "mariopedemonte12/chineutro-gemma4"   # repo privado con datos, checkpoints y resultados; None = solo local
DIR_TRABAJO = None   # None = salidas/gemma4 dentro del repo git, o /content/chineutro en Colab
MODELO_BASE = "google/gemma-4-E4B"   # base pre-entrenado (no la variante -it)
PRESUPUESTO_TOKENS = None            # None = el de config_paso1.json
LARGO_BLOQUE = 1024
MODULOS_PASO1 = ("mlp",)             # solo la FFN; ("atencion", "mlp") para la ablación
LORA_R, LORA_ALPHA, LORA_DROPOUT = 64, 128, 0.05
LR = 1e-4
SECUENCIAS_POR_PASO = 32             # lote efectivo: 32 x 1024 = 32k tokens por paso
FRACCION_VAL = 0.005                 # documentos reservados para medir perplejidad
EVAL_CADA = 50                       # pasos entre evaluaciones y checkpoints (~150 pasos con 5M tokens)
SEED = 42
DTYPE = "bfloat16"                   # "float32" solo para pruebas en GPUs sin bf16 (T4)
MAX_PASOS = -1                       # > 0 corta el entrenamiento en ese paso (pruebas)
# Fracción de los tokens de cada fuente que sale de textos con habla chilena (marcadores o chilenismos del
# diccionario); el resto, al azar. Solo Twitter: en mc4 y news casi no hay y lo que se detecta es ruido.
FRACCION_PRIORITARIA = {"twitter": 0.8}   # {} = muestreo al azar, como antes
# Barrido del LR: búsqueda binaria en escala log del LR más alto que no hace olvidar
LR_MIN, LR_MAX = 3e-5, 1e-3
ITERACIONES = 3          # cada iteración divide el intervalo (en log) a la mitad: ~x1.5 de precisión final
PASOS_ENSAYO = 40        # ~1.3M tokens por ensayo, con su propio calentamiento y coseno completos
UMBRAL_OLVIDO = 0.02     # aumento relativo máximo de la perplejidad en frases neutras (2 %)


### Datos, checkpoints y resultados: repo privado del Hugging Face Hub

Todo vive en el repo privado `REPO_HF`, que es un espejo de `DIR_TRABAJO`: `DIR_TRABAJO/x/y` queda en `x/y` del repo.

| Ruta en el repo | Contenido |
|---|---|
| `datos/` | `comun.py`, `lanzar.py`, los notebooks, `rag_documentos.jsonl`, `dataset_chilenismos_opus.csv`, `split_oficial.csv` |
| `indice_corpus.parquet`, `config_paso1.json`, `muestra_*.parquet`, `presencia_diccionario_*.csv` | salidas de la exploración (00) |
| `paso1/corpus_<M>M.parquet` | corpus del paso 1 ya construido |
| `paso1/<nombre>/last-checkpoint/` | último checkpoint del paso 1 (se reemplaza en cada guardado) |
| `paso1/<nombre>/adaptador/`, `metricas_paso1.json` | adaptador final y métricas |
| `paso2/<condición>/` | adaptador, predicciones y métricas de cada condición |
| `ejecuciones/` | notebook ejecutado y log de cada corrida sin interfaz (`lanzar.py`) |

**Sin interfaz** (runtime de Colab por API): `python datos/lanzar.py <notebook> --repo REPO_HF -p CONDICION paso1+2`. Corre el notebook con papermill y sube el log cada 10 minutos.

**Dentro del repo git** (local) se usan los datos del repo y `DIR_TRABAJO = salidas/gemma4`. Con `REPO_HF = None` no se sube ni se baja nada.

El token de Hugging Face (variable `HF_TOKEN`, `hf auth login` o el secreto `HF_TOKEN` de Colab) necesita **permiso de escritura** para subir resultados, y acceso al corpus restringido.

In [ ]:
# Entorno: token de Hugging Face, comun.py y los datos, y el repo del Hub
import os, sys, json
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

# Sin interfaz no se puede pedir el token: tiene que estar en HF_TOKEN, en el archivo de `hf auth login`
# o en los secretos de Colab
from huggingface_hub import get_token, snapshot_download
if not get_token() and EN_COLAB:
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        pass
assert get_token(), "Falta el token de Hugging Face (HF_TOKEN o hf auth login)"


def raiz_repo():
    """Raíz del repo git, si el notebook corre dentro de él."""
    actual = Path.cwd().resolve()
    return next((c for c in [actual, *actual.parents] if (c / "models/gemma4/comun.py").is_file()), None)


RAIZ_REPO = raiz_repo()
if DIR_TRABAJO is None:
    DIR_TRABAJO = RAIZ_REPO / "salidas/gemma4" if RAIZ_REPO else "/content/chineutro"
DIR_TRABAJO = Path(DIR_TRABAJO)
DIR_TRABAJO.mkdir(parents=True, exist_ok=True)

if RAIZ_REPO:   # dentro del repo git: datos y código del repo
    RAIZ, DIR_CODIGO = RAIZ_REPO, RAIZ_REPO / "models/gemma4"
else:           # Colab: comun.py y los datos vienen de datos/ en el repo del Hub
    if REPO_HF:
        snapshot_download(REPO_HF, allow_patterns=["datos/*"], local_dir=DIR_TRABAJO)
    RAIZ = DIR_CODIGO = DIR_TRABAJO / "datos"
    assert (DIR_CODIGO / "comun.py").is_file(), f"No encontré comun.py en {DIR_CODIGO}"
sys.path.insert(0, str(DIR_CODIGO))
import comun

HUB = comun.Hub(REPO_HF, DIR_TRABAJO)   # REPO_HF = None: no sube ni baja nada
for relativa in (comun.RUTA_RAG, comun.RUTA_PARES, comun.RUTA_SPLIT):
    try:
        print("✓", comun.ruta_dato(RAIZ, relativa))
    except FileNotFoundError as e:
        print(f"✗ {e}")
print("Datos:", RAIZ, "| salidas:", DIR_TRABAJO, "| Hub:", REPO_HF or "desactivado")

In [ ]:
# Estilo de gráficos: una serie en azul, grilla y ejes discretos
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

AZUL, NARANJA, TINTA, TINTA_2, GRILLA = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({"axes.edgecolor": GRILLA, "axes.labelcolor": TINTA_2, "xtick.color": TINTA_2,
                     "ytick.color": TINTA_2, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": GRILLA, "grid.linewidth": 0.6, "axes.axisbelow": True,
                     "font.size": 10, "axes.titlesize": 11})
pd.set_option("display.max_colwidth", 200)


def barras(serie, titulo, xlabel, log=False, fmt="{:,.0f}"):
    """Una serie por categoría. En escala log se dibujan puntos: el largo de una barra en log engaña."""
    s = serie.sort_values()
    fig, ax = plt.subplots(figsize=(7, 0.4 * len(s) + 1.2))
    if log:
        ax.plot(s.values, s.index.astype(str), "o", color=AZUL, markersize=8)
        ax.set_xscale("log")
        ax.grid(axis="y", visible=True)
    else:
        ax.barh(s.index.astype(str), s.values, color=AZUL, height=0.6)
        ax.grid(axis="y", visible=False)
    for y, v in enumerate(s.values):
        ax.text(v, y, " " + fmt.format(v), va="center", color=TINTA_2, fontsize=9)
    ax.set_title(titulo, loc="left", color=TINTA)
    ax.set_xlabel(xlabel)
    plt.tight_layout()
    plt.show()


def curva_perdida(historial, titulo, ylabel, guardar=None):
    """Pérdida de entrenamiento (línea) y de validación (puntos) desde trainer.state.log_history."""
    hist = pd.DataFrame(historial)
    fig, ax = plt.subplots(figsize=(7, 3.5))
    if "loss" in hist:
        tr = hist.dropna(subset=["loss"])
        ax.plot(tr.step, tr.loss, color=AZUL, linewidth=2, label="entrenamiento")
    if "eval_loss" in hist:
        ev = hist.dropna(subset=["eval_loss"])
        ax.plot(ev.step, ev.eval_loss, "o", color=NARANJA, markersize=8, markeredgecolor="white", label="validación")
    ax.set_xlabel("paso")
    ax.set_ylabel(ylabel)
    ax.set_title(titulo, loc="left", color=TINTA)
    ax.legend(frameon=False)
    plt.tight_layout()
    if guardar:
        fig.savefig(guardar, dpi=150)
    plt.show()

In [ ]:
# Mezcla de fuentes y filtros que dejó el notebook de exploración
HUB.bajar("config_paso1.json")
with open(DIR_TRABAJO / "config_paso1.json", encoding="utf-8") as f:
    CFG = json.load(f)
PRESUPUESTO_TOKENS = PRESUPUESTO_TOKENS or CFG["presupuesto_tokens"]
print(json.dumps(CFG, indent=2, ensure_ascii=False))

PRIO = "".join(f"_prio{f[:2]}{round(100 * v)}" for f, v in sorted(FRACCION_PRIORITARIA.items()) if v)
NOMBRE = f"{'+'.join(MODULOS_PASO1)}_r{LORA_R}_{PRESUPUESTO_TOKENS // 10**6}M{PRIO}"
DIR_P1 = DIR_TRABAJO / "paso1" / NOMBRE
DIR_P1.mkdir(parents=True, exist_ok=True)
print("Salidas del paso 1:", DIR_P1)

## 1. Corpus (el mismo del paso 1: si ya se construyó, se baja del Hub)

In [ ]:
from huggingface_hub import HfFileSystem
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODELO_BASE)
fs = HfFileSystem()
RUTA_CORPUS = DIR_TRABAJO / "paso1" / f"corpus_{PRESUPUESTO_TOKENS // 10**6}M{PRIO}.parquet"
HUB.bajar(RUTA_CORPUS)
if RUTA_CORPUS.exists():
    corpus = pd.read_parquet(RUTA_CORPUS)
else:
    RUTA_INDICE = DIR_TRABAJO / "indice_corpus.parquet"
    HUB.bajar(RUTA_INDICE)
    if RUTA_INDICE.exists():
        indice = pd.read_parquet(RUTA_INDICE)
    else:
        indice = comun.indice_corpus(*comun.listar_parquets(fs=fs))
        indice.to_parquet(RUTA_INDICE)
    priorizador = None
    if PRIO:
        # Qué expresiones del diccionario cuentan como habla chilena: se decide con la muestra de la exploración
        RUTA_MUESTRA = DIR_TRABAJO / f"muestra_{CFG['n_por_fuente_muestra']}.parquet"
        HUB.bajar(RUTA_MUESTRA)
        det = comun.DetectorChilenismos(comun.ruta_dato(RAIZ, comun.RUTA_RAG))
        claves, tabla_claves = comun.claves_coloquiales(pd.read_parquet(RUTA_MUESTRA), det)
        tabla_claves.to_csv(DIR_TRABAJO / "paso1" / "claves_coloquiales.csv", index=False)
        HUB.subir(DIR_TRABAJO / "paso1" / "claves_coloquiales.csv")
        una = tabla_claves[tabla_claves.cuenta & ~tabla_claves.varias_palabras]
        print(f"Expresiones que cuentan: {len(claves)} ({len(una)} de una palabra: {', '.join(una.clave.head(20))}, ...)")
        priorizador = comun.Priorizador(det, claves)
    corpus = comun.construir_corpus_paso1(indice, fs, tok, CFG["pesos"], PRESUPUESTO_TOKENS,
                                          max_caracteres=CFG["max_caracteres"],
                                          min_palabras=CFG["min_palabras"], seed=SEED,
                                          priorizador=priorizador, fraccion_prioritaria=FRACCION_PRIORITARIA)
    corpus.to_parquet(RUTA_CORPUS)
    HUB.subir(RUTA_CORPUS)

if "prioritario" not in corpus:
    corpus["prioritario"] = False
resumen = corpus.groupby(["source", "prioritario"]).n_tokens.agg(textos="count", tokens="sum")
resumen["%_tokens"] = (100 * resumen.tokens / resumen.tokens.sum()).round(1)
display(resumen)
print("Ejemplos prioritarios:")
for t in corpus[corpus.prioritario].text.sample(min(8, int(corpus.prioritario.sum())), random_state=SEED):
    print("  -", t[:160].replace("\n", " "))

In [ ]:
from datasets import Dataset

corpus = corpus.sample(frac=1.0, random_state=SEED).reset_index(drop=True)   # mezcla fuentes dentro de cada bloque
n_val = max(50, int(len(corpus) * FRACCION_VAL))
corpus_val, corpus_tr = corpus.iloc[:n_val], corpus.iloc[n_val:]


def a_bloques(df):
    return Dataset.from_pandas(df[["text"]], preserve_index=False).map(
        lambda b: comun.empaquetar(b, tok, LARGO_BLOQUE), batched=True, batch_size=1000,
        remove_columns=["text"], num_proc=4)


ds_train = a_bloques(corpus_tr)
ds_val = a_bloques(corpus_val)
ds_val = ds_val.select(range(min(200, len(ds_val))))
print(f"train: {len(ds_train):,} bloques = {len(ds_train) * LARGO_BLOQUE / 1e6:.1f}M tokens | val: {len(ds_val)} bloques")
print(tok.decode(ds_train[0]["input_ids"][:120]))

## 2. Modelo base y línea base

In [ ]:
import gc, math
from transformers import Trainer, TrainingArguments, default_data_collator

modelo = comun.cargar_modelo(MODELO_BASE, dtype=DTYPE)
pares = comun.cargar_pares(RAIZ)
val_pares = pares[pares.split == "val"]
textos_val = corpus_val.text.tolist()[:200]


def medir(m):
    return {
        "ppl_corpus_val": comun.perplejidad(m, tok, textos_val, max_len=512, bs=4),
        "ppl_frases_chilenas": comun.perplejidad(m, tok, val_pares.frase_chilena.tolist(), bs=32),
        "ppl_frases_neutras": comun.perplejidad(m, tok, val_pares.frase_neutralizada.tolist(), bs=32),
    }


DIR_B = DIR_P1 / "barrido_lr"
DIR_B.mkdir(parents=True, exist_ok=True)
HUB.bajar(DIR_B)   # retoma un barrido cortado: los ensayos ya hechos no se repiten
RUTA_ENSAYOS, RUTA_BASE = DIR_B / "ensayos.csv", DIR_B / "base.json"
BASE = json.loads(RUTA_BASE.read_text()) if RUTA_BASE.exists() else medir(modelo)
RUTA_BASE.write_text(json.dumps(BASE, indent=2))
ensayos = pd.read_csv(RUTA_ENSAYOS).to_dict("records") if RUTA_ENSAYOS.exists() else []
# Ensayos de otra corrida con otro PASOS_ENSAYO no son comparables: se descartan
ensayos = [e for e in ensayos if e.get("pasos", PASOS_ENSAYO) == PASOS_ENSAYO]
print("Línea base:", {k: round(v, 2) for k, v in BASE.items()}, "| ensayos previos:", len(ensayos))

## 3. Ensayos y búsqueda binaria

In [ ]:
POR_GPU = 8 if VRAM_GB > 70 else 4 if VRAM_GB > 35 else 1


def evaluar(r):
    # veredicto con los criterios actuales (también para ensayos reutilizados de una corrida anterior)
    finito = all(math.isfinite(r[c]) for c in ("ppl_corpus_val", "ppl_frases_chilenas", "ppl_frases_neutras"))
    r["aceptable"] = bool(finito and r["mejora_corpus_%"] > 0 and r["olvido_neutras_%"] <= 100 * UMBRAL_OLVIDO)
    return r


def ensayo(lr):
    global modelo
    previo = next((e for e in ensayos if math.isclose(e["lr"], lr, rel_tol=1e-6)), None)
    if previo:
        print(f"LR {lr:.2e}: ya ensayado -> {'aceptable' if evaluar(previo)['aceptable'] else 'NO aceptable'}")
        return previo
    k = len(ensayos) + 1
    torch.manual_seed(SEED)
    m = comun.agregar_lora(modelo, MODULOS_PASO1, LORA_R, LORA_ALPHA, LORA_DROPOUT)
    args = TrainingArguments(
        output_dir=str(DIR_B / "tmp"), per_device_train_batch_size=POR_GPU,
        gradient_accumulation_steps=max(1, SECUENCIAS_POR_PASO // POR_GPU), max_steps=PASOS_ENSAYO,
        learning_rate=lr, lr_scheduler_type="cosine", warmup_steps=0.1, weight_decay=0.0, max_grad_norm=1.0,
        bf16=DTYPE == "bfloat16", gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        logging_steps=5, save_strategy="no", eval_strategy="no", report_to="none", seed=SEED,
        dataloader_num_workers=2)
    tr = Trainer(model=m, args=args, train_dataset=ds_train, data_collator=default_data_collator,
                 callbacks=[comun.callback_progreso(extra={"ensayo": k, "lr": lr})])
    tr.train()
    r = {"lr": lr, "pasos": PASOS_ENSAYO, **medir(m),
         "loss_final": next((h["loss"] for h in reversed(tr.state.log_history) if "loss" in h), float("nan"))}
    modelo = m.unload()   # quita la LoRA sin fusionarla: el modelo vuelve a ser el base
    del tr, m
    gc.collect()
    torch.cuda.empty_cache()

    r["mejora_corpus_%"] = 100 * (1 - r["ppl_corpus_val"] / BASE["ppl_corpus_val"])
    r["mejora_chilenas_%"] = 100 * (1 - r["ppl_frases_chilenas"] / BASE["ppl_frases_chilenas"])
    r["olvido_neutras_%"] = 100 * (r["ppl_frases_neutras"] / BASE["ppl_frases_neutras"] - 1)
    evaluar(r)
    ensayos.append(r)
    pd.DataFrame(ensayos).to_csv(RUTA_ENSAYOS, index=False)
    HUB.subir(DIR_B, ignorar=["tmp/*"], mensaje=f"Barrido LR: ensayo {k} (lr {lr:.2e})")
    print(f"Ensayo {k} | LR {lr:.2e} | corpus -{r['mejora_corpus_%']:.1f} % | chilenas -{r['mejora_chilenas_%']:.1f} % "
          f"| neutras {r['olvido_neutras_%']:+.1f} % -> {'aceptable' if r['aceptable'] else 'NO aceptable'}")
    return r


for e in ensayos:   # ensayos de una corrida anterior: veredicto con los criterios actuales
    evaluar(e)
lo, hi = math.log10(LR_MIN), math.log10(LR_MAX)
LR_ELEGIDO = None
if ensayo(10 ** hi)["aceptable"]:
    LR_ELEGIDO = 10 ** hi
    print("LR_MAX ya es aceptable: sube LR_MAX si quieres explorar más arriba")
else:
    for _ in range(ITERACIONES):
        medio = (lo + hi) / 2
        if ensayo(10 ** medio)["aceptable"]:
            lo, LR_ELEGIDO = medio, 10 ** medio
        else:
            hi = medio
    if LR_ELEGIDO is None:
        if ensayo(10 ** lo)["aceptable"]:
            LR_ELEGIDO = 10 ** lo
        else:
            print("Ni LR_MIN es aceptable: baja LR_MIN o revisa UMBRAL_OLVIDO")

## 4. Resultado

In [ ]:
tabla = pd.DataFrame(ensayos).sort_values("lr")
display(tabla[["lr", "mejora_corpus_%", "mejora_chilenas_%", "olvido_neutras_%", "loss_final", "aceptable"]].round(3))

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(tabla.lr, tabla["mejora_chilenas_%"], "o-", color=AZUL, linewidth=2, label="baja de perplejidad, frases chilenas")
ax.plot(tabla.lr, tabla["olvido_neutras_%"], "o-", color=NARANJA, linewidth=2, label="alza de perplejidad, frases neutras")
ax.axhline(100 * UMBRAL_OLVIDO, color=TINTA_2, linestyle="--", linewidth=1)
ax.text(tabla.lr.min(), 100 * UMBRAL_OLVIDO, " umbral de olvido", va="bottom", color=TINTA_2, fontsize=9)
if LR_ELEGIDO:
    ax.axvline(LR_ELEGIDO, color=TINTA_2, linewidth=1)
ax.set_xscale("log")
ax.set_xlabel("learning rate")
ax.set_ylabel("cambio de perplejidad (%)")
ax.set_title(f"Barrido del LR del paso 1 ({PASOS_ENSAYO} pasos por ensayo)", loc="left", color=TINTA)
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
fig.savefig(DIR_B / "barrido_lr.png", dpi=150)
plt.show()

LR_RECOMENDADO = float(f"{LR_ELEGIDO:.1g}") if LR_ELEGIDO else None
resultado = {"lr_recomendado": LR_RECOMENDADO, "lr_elegido": LR_ELEGIDO, "umbral_olvido": UMBRAL_OLVIDO,
             "pasos_ensayo": PASOS_ENSAYO, "corpus": RUTA_CORPUS.name, "base": BASE, "ensayos": ensayos}
(DIR_B / "resultado.json").write_text(json.dumps(resultado, indent=2, ensure_ascii=False))
HUB.subir(DIR_B, ignorar=["tmp/*"], mensaje="Barrido LR: resultado")
print(f"LR recomendado para el paso 1: {LR_RECOMENDADO}  ->  lanzar.py 01_paso1_ntp_corpus.ipynb -y LR {LR_RECOMENDADO}")